In [ ]:
!pip -q uninstall -y numpy pandas scikit-learn
!pip -q install --no-cache-dir --force-reinstall numpy==1.26.4 pandas==2.2.2 scikit-learn==1.4.2 joblib gradio kagglehub

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, cross_val_score, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.ensemble import HistGradientBoostingRegressor
import joblib

In [3]:

csv_path = "/content/insurance (1).csv"

df = pd.read_csv(csv_path)

print("CSV path:", csv_path)
display(df.head())
print("Shape:", df.shape)

CSV path: /content/insurance (1).csv


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


Shape: (1338, 7)


In [4]:
df = df.copy()

#  Standardize column names
df.columns = [c.strip().lower() for c in df.columns]

# Basic sanity checks for expected columns
expected = {"age","sex","bmi","children","smoker","region","charges"}
missing_cols = expected - set(df.columns)
if missing_cols:
    raise ValueError(f"Dataset columns differ from expected. Missing: {missing_cols}. Found: {df.columns.tolist()}")

# Drop duplicates
before = len(df)
df = df.drop_duplicates()
print(f" Dropped duplicates: {before - len(df)}")

#Fix dtypes + normalize text categories
for col in ["sex","smoker","region"]:
    df[col] = df[col].astype(str).str.strip().str.lower()

df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["bmi"] = pd.to_numeric(df["bmi"], errors="coerce")
df["children"] = pd.to_numeric(df["children"], errors="coerce")
df["charges"] = pd.to_numeric(df["charges"], errors="coerce")

# Missing values
print("Missing values:\n", df.isna().sum())

# Outlier handling
num_cols_for_clip = ["age", "bmi", "children"]
for col in num_cols_for_clip:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    df[col] = df[col].clip(low, high)

# Feature engineering
def bmi_group(bmi):
    if pd.isna(bmi):
        return np.nan
    if bmi < 18.5:
        return "underweight"
    if bmi < 25:
        return "normal"
    if bmi < 30:
        return "overweight"
    return "obese"

df["bmi_group"] = df["bmi"].apply(bmi_group)
df["age_squared"] = df["age"] ** 2
df["bmi_x_smoker"] = df["bmi"] * (df["smoker"] == "yes").astype(float)

display(df.head())
print("Shape after preprocessing:", df.shape)

 Dropped duplicates: 1
Missing values:
 age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64


,age,sex,bmi,children,smoker,region,charges,bmi_group,age_squared,bmi_x_smoker
0,19,female,27.900,0,yes,southwest,16884.92400,overweight,361,27.9
1,18,male,33.770,1,no,southeast,1725.55230,obese,324,0.0
2,28,male,33.000,3,no,southeast,4449.46200,obese,784,0.0
3,33,male,22.705,0,no,northwest,21984.47061,normal,1089,0.0
4,32,male,28.880,0,no,northwest,3866.85520,overweight,1024,0.0


Shape after preprocessing: (1337, 10)


In [5]:
#train-test split
from sklearn.model_selection import train_test_split

X = df.drop(columns=["charges"])
y = df["charges"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (1069, 9) Test: (268, 9)


In [7]:
#Pipeline + Primary model selection

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor

numeric_features = ["age", "bmi", "children", "age_squared", "bmi_x_smoker"]
categorical_features = ["sex", "smoker", "region", "bmi_group"]

num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, numeric_features),
        ("cat", cat_pipe, categorical_features),
    ],
    remainder="drop"
)

base = HistGradientBoostingRegressor(random_state=42)

# target transform:
model = TransformedTargetRegressor(
    regressor=base,
    func=np.log1p,
    inverse_func=np.expm1
)

pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model)
])

pipe

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'bmi', 'children',
                                                   'age_squared',
                                                   'bmi_x_smoker']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['sex', 'smoker', 'region',
                                                   'bmi_group'])])),
                ('model',
                 TransformedTargetRegressor(func=<ufunc 'log1p'>,
                                            inverse_func=<ufunc 'expm1'>,
                                            regressor=HistGradientBoostingRegressor(random_state=42)))])

In [9]:
#train model
pipe.fit(X_train, y_train)


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'bmi', 'children',
                                                   'age_squared',
                                                   'bmi_x_smoker']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['sex', 'smoker', 'region',
                                                   'bmi_group'])])),
                ('model',
                 TransformedTargetRegressor(func=<ufunc 'log1p'>,
                                            inverse_func=<ufunc 'expm1'>,
                                            regressor=HistGradientBoostingRegressor(random_state=42)))])

In [10]:
#Cross-validation (avg + std)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

r2 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="r2")
rmse = -cross_val_score(pipe, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error")

print(f"CV R2   : {r2.mean():.4f} ± {r2.std():.4f}")
print(f"CV RMSE : {rmse.mean():.2f} ± {rmse.std():.2f}")

CV R2   : 0.8266 ± 0.0264
CV RMSE : 4825.86 ± 149.82


In [11]:
#Hyperparameter tuning

In [20]:
param_dist = {
    "model__regressor__max_depth": [2, 3, 4, 5, None],
    "model__regressor__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__regressor__max_iter": [200, 400, 800, 1200],
    "model__regressor__min_samples_leaf": [10, 20, 30, 50],
    "model__regressor__l2_regularization": [0.0, 0.1, 0.5, 1.0],
}

search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=25,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV RMSE:", -search.best_score_)

results = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
display(results[["rank_test_score", "mean_test_score", "std_test_score", "params"]].head(10))

Fitting 5 folds for each of 25 candidates, totalling 125 fits
Best params: {'model__regressor__min_samples_leaf': 30, 'model__regressor__max_iter': 200, 'model__regressor__max_depth': 5, 'model__regressor__learning_rate': 0.03, 'model__regressor__l2_regularization': 0.5}
Best CV RMSE: 4643.141220646108


,rank_test_score,mean_test_score,std_test_score,params
3,1,-4643.141221,185.333100,"{'model__regressor__min_samples_leaf': 30, 'mo..."
23,2,-4650.352832,201.102797,"{'model__regressor__min_samples_leaf': 10, 'mo..."
7,3,-4659.769332,185.323580,"{'model__regressor__min_samples_leaf': 10, 'mo..."
1,4,-4662.330404,170.939279,"{'model__regressor__min_samples_leaf': 50, 'mo..."
9,5,-4666.742450,163.423416,"{'model__regressor__min_samples_leaf': 30, 'mo..."
13,6,-4679.629827,199.226769,"{'model__regressor__min_samples_leaf': 10, 'mo..."
5,7,-4681.726276,160.060408,"{'model__regressor__min_samples_leaf': 30, 'mo..."
4,8,-4688.048702,177.180002,"{'model__regressor__min_samples_leaf': 50, 'mo..."
16,9,-4705.031281,192.286418,"{'model__regressor__min_samples_leaf': 50, 'mo..."
2,10,-4777.320490,179.545160,"{'model__regressor__min_samples_leaf': 50, 'mo..."


In [13]:
best_model = search.best_estimator_
best_model

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'bmi', 'children',
                                                   'age_squared',
                                                   'bmi_x_smoker']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['sex', 'smoker', 'region',
                                                   'bmi_group'])])),
                ('model',
                 TransformedTargetRegressor(func=<ufunc 'log1p'>,
                                            inverse_func=<ufunc 'expm1'>,
                                            regressor=HistGradientBoostingRegressor(l2_regularization=0.5,
                                                                                    learning_rate=0.03,
                                                                                    max_depth=5,
                                                                                    max_iter=200,
                                                                                    min_samples_leaf=30,
                                                                                    random_state=42)))])

In [14]:
#test performance evaluation

y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

print("===== Test Metrics =====")
print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2  : {r2:.4f}")

===== Test Metrics =====
MAE : 2048.26
RMSE: 4309.19
R2  : 0.8989


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [21]:
#save best pipeline

import joblib

os.makedirs("artifacts", exist_ok=True)
MODEL_PATH = "artifacts/insurance_best_pipeline.joblib"
joblib.dump(best_model, MODEL_PATH)
print("Saved:", MODEL_PATH)

Saved: artifacts/insurance_best_pipeline.joblib


In [16]:
import gradio as gr

loaded = joblib.load(MODEL_PATH)

def predict_charges(age, sex, bmi, children, smoker, region):
    row = pd.DataFrame([{
        "age": age,
        "sex": str(sex).strip().lower(),
        "bmi": bmi,
        "children": children,
        "smoker": str(smoker).strip().lower(),
        "region": str(region).strip().lower(),
    }])

    # Must match training feature engineering
    def bmi_group_local(b):
        if b < 18.5: return "underweight"
        if b < 25:   return "normal"
        if b < 30:   return "overweight"
        return "obese"

    row["bmi_group"] = row["bmi"].apply(bmi_group_local)
    row["age_squared"] = row["age"] ** 2
    row["bmi_x_smoker"] = row["bmi"] * (row["smoker"] == "yes").astype(float)

    return float(loaded.predict(row)[0])

demo = gr.Interface(
    fn=predict_charges,
    inputs=[
        gr.Number(label="Age", value=30),
        gr.Dropdown(["female","male"], label="Sex", value="male"),
        gr.Number(label="BMI", value=26.0),
        gr.Number(label="Children", value=0),
        gr.Dropdown(["yes","no"], label="Smoker", value="no"),
        gr.Dropdown(["northeast","northwest","southeast","southwest"], label="Region", value="southeast"),
    ],
    outputs=gr.Number(label="Predicted Insurance Charges"),
    title="Medical Insurance Cost Prediction",
    description="Enter values and get predicted insurance charges."
)

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6e4fc5f5775111a213.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Created dataset file at: .gradio/flagged/dataset1.csv
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://6e4fc5f5775111a213.gradio.live
